In [1]:
import requests
import pandas as pd
import geopandas as gpd

## Dataset 1: Complaints

Collect complaints from NYC Open Data <br> 
(https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/about_data)

- from the Department of Housing Preservation and Development (HPD)
- from 1 january 2021 to 31 december 2025 (created_date column)
- complaint types selected
    - Heat/Hot water
    - Unsanitary condition
    - Water leak
    - Plumbing
    - Electric
- complaints along with zip code and borough
- 2.468.069 complaints

In [ ]:
# base_url_complaints = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

# params_complaints = {
#     "$select": "unique_key, created_date, agency_name, complaint_type, incident_zip, borough",
#     "$where": (
#         "agency = 'HPD' "
#         "AND created_date >= '2021-01-01T00:00:00.000' "
#         "AND created_date < '2026-01-01T00:00:00.000' "
#         "AND complaint_type != 'GENERAL' "
#         "AND complaint_type != 'General' "
#         "AND complaint_type != 'ELEVATOR' "
#         "AND complaint_type != 'Elevator' "
#         "AND complaint_type != 'DOOR/WINDOW' "
#         "AND complaint_type != 'Door/Window' "
#         "AND complaint_type != 'PAINT/PLASTER' "
#         "AND complaint_type != 'Paint/Plaster' "
#         "AND complaint_type != 'APPLIANCE' "
#         "AND complaint_type != 'Appliance' "
#         "AND complaint_type != 'FLOORING/STAIRS' "
#         "AND complaint_type != 'Flooring/Stairs' "
#         "AND complaint_type != 'OUTSIDE BUILDING' "
#         "AND complaint_type != 'Outside Building' "
#         "AND complaint_type != 'SAFETY' "
#         "AND complaint_type != 'Safety' "

#     ),
#     "$limit": 10000000
# }

# response = requests.get(base_url_complaints, params=params_complaints)
# data = response.json()
# df_complaints = pd.DataFrame(data)
# df_complaints.head(10)

,unique_key,created_date,agency_name,complaint_type,incident_zip,borough
0,59486769,2023-11-18T20:52:30.000,Department of Housing Preservation and Develop...,HEAT/HOT WATER,10461,BRONX
1,59479370,2023-11-17T17:58:35.000,Department of Housing Preservation and Develop...,UNSANITARY CONDITION,11385,QUEENS
2,59484945,2023-11-18T09:02:27.000,Department of Housing Preservation and Develop...,WATER LEAK,10473,BRONX
3,59483900,2023-11-18T16:36:35.000,Department of Housing Preservation and Develop...,WATER LEAK,10037,MANHATTAN
4,59484864,2023-11-18T07:49:47.000,Department of Housing Preservation and Develop...,HEAT/HOT WATER,10039,MANHATTAN
5,59476772,2023-11-17T18:04:52.000,Department of Housing Preservation and Develop...,HEAT/HOT WATER,10463,BRONX
6,59478062,2023-11-17T20:18:38.000,Department of Housing Preservation and Develop...,PLUMBING,10031,MANHATTAN
7,59478051,2023-11-17T12:09:15.000,Department of Housing Preservation and Develop...,PLUMBING,10456,BRONX
8,59481731,2023-11-18T19:19:41.000,Department of Housing Preservation and Develop...,HEAT/HOT WATER,10036,MANHATTAN
9,59478100,2023-11-17T13:56:48.000,Department of Housing Preservation and Develop...,UNSANITARY CONDITION,11207,BROOKLYN


In [11]:
print("Missing values per column")
print(df_complaints.isna().sum())

Missing values per column
unique_key          0
created_date        0
agency_name         0
complaint_type      0
incident_zip      334
borough             4
dtype: int64


In [ ]:
df_complaints = df_complaints.dropna()
print(f"Number of rows after dropping NaN: {len(df_complaints)}" )

Number of rows after dropping NaN: 2468069


In [13]:
print(f"Number of observations: {len(df_complaints)}")

print("Diferent complaint types:")
df_complaints["complaint_type"] = df_complaints["complaint_type"].str.upper()
print(df_complaints["complaint_type"].unique())

print(f"Number of unique incident zip codes: {len(df_complaints['incident_zip'].unique())}")

Number of observations: 2468069
Diferent complaint types:
<StringArray>
['HEAT/HOT WATER', 'UNSANITARY CONDITION', 'WATER LEAK', 'PLUMBING',
 'ELECTRIC']
Length: 5, dtype: str
Number of unique incident zip codes: 211


In [14]:
df_complaints["created_date"]=pd.to_datetime(df_complaints["created_date"])
min_date=df_complaints['created_date'].min()
max_date=df_complaints['created_date'].max()
print(f"Date range: {min_date}, {max_date}")

Date range: 2021-01-01 00:01:14, 2025-12-31 23:57:56


In [15]:
df_complaints_copy = df_complaints.sort_values("created_date", ascending=False)
df_complaints_copy.to_csv('data/complaints20_25_original.csv', index=False)

## Dataset 2: Violations

Collect violations from NYC Open Data <br> 
(https://data.cityofnewyork.us/Housing-Development/Housing-Maintenance-Code-Violations/wvxf-dwi5/about_data)

- Inspection date: date when violation was observed
- Certified date: date when violation was certified
- Violation Status: 
    - An **OPEN** violation is a violation which is still active on the Department records. The property owner may or may not have corrected the physical condition if the status is open. 
    - A **CLOSE** violation is observed/verified as corrected by HPD or as certified by the landlord.

- from 1 january 2021 to 31 december 2025 (inspection date)
- violation classes
    - Class A (non-hazardous)
    - Class B (hazardous)
    - Class C (immediately hazardous)
- 3.993.658 violations
- violations along with zip code, borough and registration ID

In [15]:
base_url_violations = "https://data.cityofnewyork.us/resource/wvxf-dwi5.json"

params_violations = {
    "$select": "violationid, zip, class, inspectiondate, certifieddate, boro, violationstatus, registrationid",
    "$where": "inspectiondate >= '2021-01-01T00:00:00.000' AND inspectiondate < '2026-01-01T00:00:00.000' AND class !='I'", 
    "$limit": 10000000
}

response = requests.get(base_url_violations, params=params_violations)
data = response.json()
df_violations = pd.DataFrame(data)
df_violations.head(10)

,violationid,zip,class,inspectiondate,certifieddate,boro,violationstatus,registrationid
0,13967395,11233,C,2021-01-01T00:00:00.000,2021-02-10T00:00:00.000,BROOKLYN,Close,371310
1,13967407,11230,A,2021-01-01T00:00:00.000,NaN,BROOKLYN,Close,305293
2,13967421,11233,A,2021-01-01T00:00:00.000,2021-03-05T00:00:00.000,BROOKLYN,Close,380205
3,13967510,11226,C,2021-01-01T00:00:00.000,NaN,BROOKLYN,Close,813649
4,13967509,11226,C,2021-01-01T00:00:00.000,NaN,BROOKLYN,Close,813649
5,13967520,11226,B,2021-01-01T00:00:00.000,NaN,BROOKLYN,Close,813649
6,13967521,11226,B,2021-01-01T00:00:00.000,NaN,BROOKLYN,Close,813649
7,13967414,11230,B,2021-01-01T00:00:00.000,NaN,BROOKLYN,Close,305293
8,13967522,11226,B,2021-01-01T00:00:00.000,NaN,BROOKLYN,Close,813649
9,13894917,10025,C,2021-01-01T00:00:00.000,2024-04-10T00:00:00.000,MANHATTAN,Close,143361


In [16]:
print("Missing values per column")
print(df_violations.isna().sum())

Missing values per column
violationid              0
zip                   4369
class                    0
inspectiondate           0
certifieddate      2319898
boro                     0
violationstatus          0
registrationid           0
dtype: int64


In [17]:
df_violations = df_violations.dropna(subset=["zip"])
print(f"Number of observations after droping NaN in zip: {len(df_violations)}")

Number of observations after droping NaN in zip: 3993675


In [18]:
print(f"Number of unique incident zip codes: {len(df_violations['zip'].unique())}")

Number of unique incident zip codes: 208


In [19]:
df_violations["inspectiondate"]=pd.to_datetime(df_violations["inspectiondate"])
min_date=df_violations['inspectiondate'].min()
max_date=df_violations['inspectiondate'].max()
print(f"Date range: {min_date}, {max_date}")

Date range: 2021-01-01 00:00:00, 2025-12-31 00:00:00


In [20]:
df_violations = df_violations.sort_values("inspectiondate", ascending=False)
df_violations.to_csv('data/violations20_25_original.csv', index=False)

## Dataset 3: Income per area

2018-2022 ACS 5-Year Estimates for ZIP 10001:

Survey Year → Question Asked
────────────────────────────
2018 → "Income in past 12 months (2017-2018)?" → 500 households answer
2019 → "Income in past 12 months (2018-2019)?" → 500 households answer  
2020 → "Income in past 12 months (2019-2020)?" → 500 households answer
2021 → "Income in past 12 months (2020-2021)?" → 500 households answer
2022 → "Income in past 12 months (2021-2022)?" → 500 households answer

Census pools all 2,500 responses → calculates estimate for ZIP 10001

1. Find Available 5 Years Subject Tables

In [44]:
groups_url = "https://api.census.gov/data/2022/acs/acs5/subject/groups.json"
response = requests.get(groups_url)
groups = response.json()
groups_df = pd.DataFrame(groups['groups'])
groups_df.head(20)

,name,description,variables
0,S0103PR,Population 65 Years and Over in Puerto Rico,http://api.census.gov/data/2022/acs/acs5/subject/groups/S0103PR.json
1,S1601,Language Spoken at Home,http://api.census.gov/data/2022/acs/acs5/subject/groups/S1601.json
2,S2414,"Industry by Sex and Median Earnings in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) for the Full-Time, Year-Round Civilian Employed Population 16 Years and Over",http://api.census.gov/data/2022/acs/acs5/subject/groups/S2414.json
3,S1602,Limited English Speaking Households,http://api.census.gov/data/2022/acs/acs5/subject/groups/S1602.json
4,S0502PR,Selected Characteristics of the Foreign-Born Population by Period of Entry Into Puerto Rico,http://api.census.gov/data/2022/acs/acs5/subject/groups/S0502PR.json
5,S1603,Characteristics of People by Language Spoken at Home,http://api.census.gov/data/2022/acs/acs5/subject/groups/S1603.json
6,S2419,"Class of Worker by Sex and Median Earnings in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) for the Full-Time, Year-Round Civilian Employed Population 16 Years and Over",http://api.census.gov/data/2022/acs/acs5/subject/groups/S2419.json
7,S2418,Class of Worker by Sex and Median Earnings in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) for the Civilian Employed Population 16 Years and Over,http://api.census.gov/data/2022/acs/acs5/subject/groups/S2418.json
8,S1001,Grandchildren Characteristics,http://api.census.gov/data/2022/acs/acs5/subject/groups/S1001.json
9,S1002,Grandparents,http://api.census.gov/data/2022/acs/acs5/subject/groups/S1002.json


2. Income related tables have S19

In [45]:
income_tables = groups_df[groups_df['name'].str.contains('S19', na=False)]
print(income_tables[['name', 'description']])

     name  \
53  S1901   
54  S1902   
56  S1903   

                                                                 description  
53         Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars)  
54    Mean Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars)  
56  Median Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars)  


3. Getting all  variables in a S19 table, use mean table

S1901_C01_012E
  │    │  │  └─ E = Estimate, M = Margin of Error
  │    │  └──── Variable number (001, 002, 003...)
  │    └─────── Column: C01 = Total, C02 = Male, C03 = Female
  └──────────── Table S1901

In [46]:
table_url = "https://api.census.gov/data/2024/acs/acs5/subject/groups/S1901.json"
response = requests.get(table_url)
table_info = response.json()

variables = []
for var_name, var_info in table_info['variables'].items():
    if var_name != 'for' and var_name != 'in':  # Skip geography variables
        variables.append({
            'variable': var_name,
            'label': var_info.get('label', ''),
            'concept': var_info.get('concept', '')
        })

vars_df = pd.DataFrame(variables)
print(f"Total variables in S1901: {len(vars_df)}")
vars_df.head(10)

Total variables in S1901: 258


,variable,label,concept
0,S1901_C04_009EA,"Annotation of Estimate!!Nonfamily households!!Total!!$100,000 to $149,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
1,S1901_C04_009MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$100,000 to $149,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
2,S1901_C04_008EA,"Annotation of Estimate!!Nonfamily households!!Total!!$75,000 to $99,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
3,S1901_C04_008MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$75,000 to $99,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
4,S1901_C04_007EA,"Annotation of Estimate!!Nonfamily households!!Total!!$50,000 to $74,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
5,S1901_C04_007MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$50,000 to $74,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
6,S1901_C04_006EA,"Annotation of Estimate!!Nonfamily households!!Total!!$35,000 to $49,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
7,S1901_C04_006MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$35,000 to $49,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
8,S1901_C04_005EA,"Annotation of Estimate!!Nonfamily households!!Total!!$25,000 to $34,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)
9,S1901_C04_005MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$25,000 to $34,999",Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)


In [47]:
pd.set_option('display.max_colwidth', None)  # Show full text
vars_df[['variable', 'label']].head(10)

,variable,label
0,S1901_C04_009EA,"Annotation of Estimate!!Nonfamily households!!Total!!$100,000 to $149,999"
1,S1901_C04_009MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$100,000 to $149,999"
2,S1901_C04_008EA,"Annotation of Estimate!!Nonfamily households!!Total!!$75,000 to $99,999"
3,S1901_C04_008MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$75,000 to $99,999"
4,S1901_C04_007EA,"Annotation of Estimate!!Nonfamily households!!Total!!$50,000 to $74,999"
5,S1901_C04_007MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$50,000 to $74,999"
6,S1901_C04_006EA,"Annotation of Estimate!!Nonfamily households!!Total!!$35,000 to $49,999"
7,S1901_C04_006MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$35,000 to $49,999"
8,S1901_C04_005EA,"Annotation of Estimate!!Nonfamily households!!Total!!$25,000 to $34,999"
9,S1901_C04_005MA,"Annotation of Margin of Error!!Nonfamily households!!Total!!$25,000 to $34,999"


This is the variable we want, it is median income.

In [48]:
vars_df[vars_df["variable"] == "S1901_C01_012E"]

,variable,label,concept
156,S1901_C01_012E,Estimate!!Households!!Median income (dollars),Income in the Past 12 Months (in 2024 Inflation-Adjusted Dollars)


4. Getting median income for all zip codes in new york

In [49]:
base_url = "https://api.census.gov/data/2024/acs/acs5/subject"

params = {
    "get": "NAME,S1901_C01_012E,S1901_C01_012M", #this is median
    "for": "zip code tabulation area:*"
}

print("Fetching data from Census API...")
response = requests.get(base_url, params=params)

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ['location', 'median_income', 'margin_of_error', 'zipcode']
    
    df['median_income'] = pd.to_numeric(df['median_income'], errors='coerce')
    df['margin_of_error'] = pd.to_numeric(df['margin_of_error'], errors='coerce')
    
    # filter for new york city, not new york state(5 boroughs)
    def is_nyc_zip(zipcode):
        if zipcode.startswith('100') or zipcode.startswith('101') or zipcode.startswith('102'):
            return True  # Manhattan
        elif zipcode.startswith('103'):
            return True  # Staten Island
        elif zipcode.startswith('104'):
            return True  # Bronx
        elif zipcode.startswith('110') or zipcode.startswith('111') or zipcode.startswith('113') or zipcode.startswith('114') or zipcode.startswith('116'):
            return True  # Queens
        elif zipcode.startswith('112'):
            return True  # Brooklyn
        else:
            return False
    
    # Filter for nyc
    nyc_df = df[df['zipcode'].apply(is_nyc_zip)].copy()

    # Replace Census missing-value codes with NaN
    missing_codes = [-666666666]
    nyc_df['median_income'] = nyc_df['median_income'].replace(missing_codes, pd.NA)
    nyc_df['margin_of_error'] = nyc_df['margin_of_error'].replace(missing_codes, pd.NA)
    nyc_df = nyc_df[nyc_df['median_income'].notna()]
    nyc_df = nyc_df.sort_values('median_income', ascending=False).reset_index(drop=True)
    
    # Add borough column
    def get_borough(zipcode):
        if zipcode.startswith('100') or zipcode.startswith('101') or zipcode.startswith('102'):
            return 'Manhattan'
        elif zipcode.startswith('103'):
            return 'Staten Island'
        elif zipcode.startswith('104'):
            return 'Bronx'
        elif zipcode.startswith('110') or zipcode.startswith('111') or zipcode.startswith('113') or zipcode.startswith('114') or zipcode.startswith('116'):
            return 'Queens'
        elif zipcode.startswith('112'):
            return 'Brooklyn'
        else:
            return 'Unknown'
    
    nyc_df['borough'] = nyc_df['zipcode'].apply(get_borough)
    
    print(f"Total NYC ZIP codes: {len(nyc_df)}")

    nyc_df.to_csv('data/nyc_zipcode_income.csv', index=False)
    print(f"\n Saved to 'nyc_zipcode_income.csv'")
    
else:
    print(f"Error {response.status_code}: {response.text}")

Fetching data from Census API...
Total NYC ZIP codes: 192

 Saved to 'nyc_zipcode_income.csv'


In [50]:
# Borough statistics
print(f"ZIP Codes by Borough")
print(nyc_df['borough'].value_counts().sort_index())

print(f"\nTop 10 highest income ZIP codes in NYC:")
print(nyc_df[['zipcode', 'borough', 'median_income']].head(10))

print(f"\nBottom 10 lowest income ZIP codes in NYC:")
print(nyc_df[['zipcode', 'borough', 'median_income']].tail(10))

print(f"\nMedian Income by Borough")
borough_stats = nyc_df.groupby('borough')['median_income'].agg(['mean', 'median', 'min', 'max', 'count'])
borough_stats.columns = ['Average', 'Median', 'Lowest', 'Highest', 'ZIP_Count']
print(borough_stats.round(0))


ZIP Codes by Borough
borough
Bronx            25
Brooklyn         38
Manhattan        46
Queens           71
Staten Island    12
Name: count, dtype: int64

Top 10 highest income ZIP codes in NYC:
  zipcode    borough median_income
0   10282  Manhattan        250001
1   10004  Manhattan        250001
2   10007  Manhattan        250001
3   10279  Manhattan        250001
4   11030     Queens        234966
5   10069  Manhattan        233636
6   10280  Manhattan        206299
7   11109     Queens        204885
8   10005  Manhattan        190233
9   10006  Manhattan        190170

Bottom 10 lowest income ZIP codes in NYC:
    zipcode    borough median_income
182   10030  Manhattan         39802
183   10451      Bronx         38770
184   10029  Manhattan         38695
185   11239   Brooklyn         38015
186   10459      Bronx         38006
187   10460      Bronx         36309
188   10455      Bronx         36115
189   10456      Bronx         34954
190   10453      Bronx         33186
191   

In [51]:
# NYC statistics
print(f"\nOverall NYC Income Statistics")
print(f"Highest: ${nyc_df['median_income'].max():,.0f}")
print(f"Lowest: ${nyc_df['median_income'].min():,.0f}")
print(f"Average: ${nyc_df['median_income'].mean():,.0f}")
print(f"Median: ${nyc_df['median_income'].median():,.0f}")


Overall NYC Income Statistics
Highest: $250,001
Lowest: $24,086
Average: $100,357
Median: $90,394


## APAGAR: Multiple Dwelling Registration

Registration information from owners of residential rental units

In [32]:
base_url_complaints = "https://data.cityofnewyork.us/resource/tesw-yqqr.json"

params_complaints = {
    "$select": "registrationid, buildingid, boro, zip, lastregistrationdate",
    "$where": "lastregistrationdate >= '2021-01-01T00:00:00.000'",
    "$limit": 10000000
}

response = requests.get(base_url_complaints, params=params_complaints)
data = response.json()
df_registrations = pd.DataFrame(data)
df_registrations.head(10)

,registrationid,buildingid,boro,zip,lastregistrationdate
0,825850,271859,BROOKLYN,11234,2026-03-19T00:00:00.000
1,131572,169,MANHATTAN,10003,2025-08-13T00:00:00.000
2,109808,245,MANHATTAN,10028,2025-08-08T00:00:00.000
3,420463,810843,QUEENS,11434,2026-03-31T00:00:00.000
4,335058,383549,BROOKLYN,11231,2026-03-30T00:00:00.000
5,125763,1043,MANHATTAN,10128,2025-07-10T00:00:00.000
6,137997,1699,MANHATTAN,10021,2025-07-11T00:00:00.000
7,142492,8531,MANHATTAN,10003,2026-03-19T00:00:00.000
8,124742,2193,MANHATTAN,10010,2025-08-11T00:00:00.000
9,110059,2635,MANHATTAN,10035,2025-08-24T00:00:00.000


In [33]:
print("Missing values per column")
print(df_registrations.isna().sum())

Missing values per column
registrationid          0
buildingid              0
boro                    0
zip                     7
lastregistrationdate    0
dtype: int64


In [34]:
print(f"Number of observations: {len(df_registrations)}")
print(f"Number of unique registrationID: {len(df_registrations['registrationid'].unique())}")
print(f"Number of unique buildingID : {len(df_registrations['buildingid'].unique())}")
print(f"Number of unique boroughs: {len(df_registrations['boro'].unique())}")

Number of observations: 176076
Number of unique registrationID: 166423
Number of unique buildingID : 176070
Number of unique boroughs: 5


In [35]:
df_registrations.to_csv('data/registrations_original.csv', index=False)

## Dataset 4: RegistrationID contact

In [21]:
base_url_complaints = "https://data.cityofnewyork.us/resource/feu5-w2e2.json"

params_complaints = {
    "$select": "registrationcontactid, registrationid, type, corporationname, businessstreetname, businesshousenumber, firstname, lastname",
    "$limit": 10000000
}

response = requests.get(base_url_complaints, params=params_complaints)
data = response.json()
df_contacts = pd.DataFrame(data)
df_contacts.head(10)

,registrationcontactid,registrationid,type,corporationname,businessstreetname,businesshousenumber,firstname,lastname
0,12473203,124732,CorporateOwner,KOSCAL 59 LLC,3 59TH ST,346,NaN,NaN
1,12473204,124732,Agent,TZM REALTY PROPERTIES LLC,E 59TH ST,346,ANTHONY,ZACHARIADIS
2,12473205,124732,HeadOfficer,NaN,E 59TH ST,346,KOSTAS,ZACHARIADIS
3,12473206,124732,Officer,NaN,E 59TH ST,346,ANTHONY,ZAHARIADIS
4,12473213,124732,SiteManager,NaN,NaN,NaN,ROSALIO,PEREA
5,13157203,131572,CorporateOwner,GRAY ROCK EQUITIES LLC,MELISSA CT,207-21,NaN,NaN
6,13157204,131572,Agent,REGAL PROPERTY MANAGEMENT INC.,WEST 25 STREET,18,CHUN,YUNG
7,13157205,131572,HeadOfficer,NaN,1ST AVE,141,YONG,KIM
8,13157206,131572,Officer,NaN,1ST AVE,141,HAN SANG,BAE
9,10980803,109808,CorporateOwner,"1576.FIRST AVENUE, INC",E 82 ST,400,NaN,NaN


In [24]:
print("Missing values per column")
print(df_contacts.isna().sum())

Missing values per column
registrationcontactid         0
registrationid                0
type                          0
corporationname          569202
businessstreetname       166201
businesshousenumber      166344
firstname                124951
lastname                 125448
dtype: int64


In [25]:
print(f"Number of observations: {len(df_contacts)}")
print(f"Number of unique registrationID: {len(df_contacts['registrationid'].unique())}")

Number of observations: 781371
Number of unique registrationID: 181976


In [3]:
df_contacts[df_contacts["registrationid"]=="911750"]

,registrationcontactid,registrationid,type,contactdescription,corporationname,businessstreetname,businesshousenumber
221130,91175002,911750,IndividualOwner,CORP,NaN,S. METROPOLITAN,715
221131,91175003,911750,CorporateOwner,CORP,HYDE PARK OWNERS CORP.,JEWEL AVENUE,137-07
221132,91175004,911750,Agent,CORP,METRO MANAGEMENT DEV.,MARCUS AVENUE,1981
221133,91175005,911750,HeadOfficer,CORP,NaN,JEWEL AVENUE,137-07
221134,91175006,911750,Officer,CORP,NaN,JEWEL AVENUE,137-07
...,...,...,...,...,...,...,...
706936,91175006,911750,Officer,CORP,NaN,JEWEL AVENUE,137-07
706937,91175006,911750,Officer,CORP,NaN,JEWEL AVENUE,137-07
706938,91175006,911750,Officer,CORP,NaN,JEWEL AVENUE,137-07
706939,91175006,911750,Officer,CORP,NaN,JEWEL AVENUE,137-07


In [26]:
df_contacts.to_csv('data/contacts_original.csv', index=False)